In [ ]:
# Bundle transformers + a few critical companion wheels for offline install
# in the dev kernel. The Kaggle H100 image (April 2026) ships transformers
# 5.0.0, which does not recognize the qwen3_5_moe architecture used by
# Qwen3.6-35B-A3B. The latest transformers (>=5.5 at time of writing) does.
#
# We grab transformers + tokenizers + accelerate + huggingface_hub + safetensors
# (and their direct deps via pip download), staging into /tmp/, then upload
# as a private Kaggle Dataset.
#
# Sequence:
#   1. pip download transformers tokenizers accelerate hf-transfer
#      huggingface_hub safetensors --dest /tmp/transformers_pkg/wheels
#   2. write dataset-metadata.json
#   3. kaggle datasets create OR kagglehub.dataset_upload
#
# Time budget: ~2-5 min for download, ~1-2 min for upload.
import os, json, sys, time, subprocess
from pathlib import Path

DATASET_OWNER = os.environ.get('KAGGLE_USER', 'cataluna84')
DATASET_SLUG  = 'transformers-qwen3-bundle'
DATASET_TITLE = 'Transformers + Qwen3 deps wheels (offline mirror)'
LOCAL_DIR     = '/tmp/transformers_pkg'
WHEELS_DIR    = f'{LOCAL_DIR}/wheels'

Path(WHEELS_DIR).mkdir(parents=True, exist_ok=True)

# --- Step 1: pip download into wheels dir -----------------------------------
print('[1/3] downloading transformers + companion wheels', flush=True)
t0 = time.time()
PACKAGES = [
    'transformers',
    'tokenizers',
    'accelerate',
    'huggingface_hub',
    'safetensors',
]
rc = subprocess.run(
    [sys.executable, '-m', 'pip', 'download',
     '--dest', WHEELS_DIR,
     '--python-version', '3.12',
     '--platform', 'manylinux_2_28_x86_64',
     '--platform', 'manylinux_2_27_x86_64',
     '--platform', 'manylinux2014_x86_64',
     '--platform', 'manylinux2010_x86_64',
     '--platform', 'any',
     '--only-binary=:all:',
     '--no-deps',
     *PACKAGES],
    capture_output=True, text=True,
)
print('rc=', rc.returncode)
print('stdout (last 1500):', rc.stdout[-1500:])
print('stderr (last 500):', rc.stderr[-500:])

# Also pull deps WITH deps in a permissive way; if some don't have manylinux
# wheels, fall back to plain pip download (lets pip pick whatever wheel is
# acceptable for cp312-linux).
rc2 = subprocess.run(
    [sys.executable, '-m', 'pip', 'download',
     '--dest', WHEELS_DIR,
     '--no-deps',
     'transformers', 'tokenizers', 'accelerate', 'huggingface_hub', 'safetensors',
     'regex', 'filelock', 'fsspec', 'pyyaml', 'tqdm',
    ],
    capture_output=True, text=True,
)
print('rc2=', rc2.returncode)
print('stdout (last 1500):', rc2.stdout[-1500:])
print('stderr (last 500):', rc2.stderr[-500:])

dl_min = (time.time() - t0) / 60.0
wheels = sorted(Path(WHEELS_DIR).glob('*.whl'))
total_bytes = sum(f.stat().st_size for f in wheels)
print(f'[1/3] done in {dl_min:.1f} min  wheels={len(wheels)}  total={total_bytes/1e6:.2f} MB')
for w in wheels:
    print(f'  {w.stat().st_size/1e6:>7.2f} MB  {w.name}')

# --- Step 2: write dataset-metadata.json ------------------------------------
meta = {
    'title': DATASET_TITLE,
    'id': f'{DATASET_OWNER}/{DATASET_SLUG}',
    'licenses': [{'name': 'apache-2.0'}],
    'subtitle': 'Transformers + companion wheels for offline qwen3_5_moe support',
    'description': (
        'Mirrored wheels of transformers + tokenizers + accelerate + '
        'huggingface_hub + safetensors (and a few common deps), bundled here '
        'so kernels with enable_internet=false can install the architectures '
        'they need without hitting PyPI. The Kaggle H100 image as of April '
        '2026 has transformers 5.0.0 which does not include qwen3_5_moe.'
    ),
    'isPrivate': True,
}
with open(f'{LOCAL_DIR}/dataset-metadata.json', 'w') as fh:
    json.dump(meta, fh, indent=2)
print('[2/3] wrote dataset-metadata.json')

# --- Step 3: upload --------------------------------------------------------
print('[3/3] uploading to Kaggle Datasets...', flush=True)
uploaded = False
t1 = time.time()
try:
    import kagglehub
    handle = f'{DATASET_OWNER}/{DATASET_SLUG}'
    h = kagglehub.dataset_upload(handle, LOCAL_DIR,
                                  version_notes='transformers wheels initial bundle')
    print(f'    kagglehub returned: {h!r}')
    uploaded = True
except Exception as e:
    print(f'    kagglehub.dataset_upload failed: {type(e).__name__}: {e}')
if not uploaded:
    print('    fallback: kaggle datasets create')
    res = subprocess.run(
        ['kaggle', 'datasets', 'create', '-p', LOCAL_DIR, '--dir-mode', 'tar'],
        capture_output=True, text=True, timeout=1200,
    )
    print('CLI stdout:', res.stdout)
    print('CLI stderr:', res.stderr)
    if res.returncode == 0:
        uploaded = True
ul_min = (time.time() - t1) / 60.0
print()
print('=== SUMMARY ===')
print(f'  wheels count   : {len(wheels)}')
print(f'  total bytes    : {total_bytes/1e6:.2f} MB')
print(f'  download time  : {dl_min:.1f} min')
print(f'  upload time    : {ul_min:.1f} min')
print(f'  Kaggle Dataset : {DATASET_OWNER}/{DATASET_SLUG}')
print(f'  uploaded ok    : {uploaded}')
